## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

>  **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 5 dimulai**.
>  **Sifat tugas:** individu.

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [4]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


### Instruksi Pengerjaan

Buat notebook baru **`Tugas4_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` baru, lalu kerjakan bagian **A sampai E** berikut — **seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

---

**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

In [7]:
df_tugas = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df_tugas.printSchema()
print("Jumlah baris:", df_tugas.count())
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [10]:
# Cek nilai null pada rating
null_count = df_tugas.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", null_count)

# Mengisi nilai null dengan rating netral 3.0
df_clean = df_tugas.na.fill({"rating": 3.0})

Jumlah rating kosong: 204


**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 


In [11]:
from pyspark.sql.functions import when

df_transformed = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.select("order_id", "total_pendapatan", "tier_transaksi").show(10)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
|ORD-3005|          100000|         Kecil|
|ORD-3006|           40000|         Kecil|
|ORD-3007|          720000|         Besar|
|ORD-3008|          140000|         Kecil|
|ORD-3009|          900000|         Besar|
+--------+----------------+--------------+
only showing top 10 rows



**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

**1. Kategori dengan total pendapatan tertinggi:**

In [13]:
from pyspark.sql.functions import col, sum as spark_sum

# Menampilkan 1 kategori dengan total pendapatan tertinggi
df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

[Stage 13:>                                                         (0 + 1) / 1]

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row



**2. Kota dengan transaksi tier "Besar" terbanyak:**

In [14]:
from pyspark.sql.functions import col, sum as spark_sum

df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota").count() \
    .orderBy(col("count").desc()) \
    .show(1)

[Stage 16:>                                                         (0 + 1) / 1]

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row



**3. Rata-rata rating per metode pembayaran:**

In [18]:
from pyspark.sql.functions import col, avg, sum as spark_sum

df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

[Stage 19:>                                                         (0 + 1) / 1]

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|3.948207171314741|
|    Transfer Bank|3.932806324110672|
|     Kartu Kredit|3.861788617886179|
|         E-Wallet|            3.904|
+-----------------+-----------------+



**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [20]:
df_transformed.write.mode("overwrite").csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transformasi_september",
    header=True
)

**Mengapa Berkas Tersimpan dalam Beberapa Partisi (part-00000...)?**

Spark memproses data secara terdistribusi memanfaatkan seluruh core CPU/executor. Saat menulis data ke penyimpanan, masing-masing executor menulis bagian datanya (partition) secara terpisah dan paralel untuk memaksimalkan kecepatan I/O.